# **COMPREHENSIVE MODEL TRAINING WITH BALANCED DATA**

## **0. Library import**

In [10]:
import os
import sys

# Add the root path into the python path
root_path = os.path.abspath(os.path.join(".."))
if not root_path in sys.path:
    sys.path.insert(0, root_path)

In [11]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

from src.config import LOG_DIR, MODEL_DIR, SCALERS_DIR, \
    RANDOM_OVER_SAMPLING_DATA_FILE_PATH, SMOTE_DATA_FILE_PATH, ADASYN_DATA_FILE_PATH, \
    RANDOM_UNDER_SAMPLING_DATA_FILE_PATH, TOMEK_LINKS_DATA_FILE_PATH, EDITED_NEIGHBORS_DATA_FILE_PATH, \
    SMOTE_TOMEK_LINKS_DATA_FILE_PATH, SMOTE_ENN_DATA_FILE_PATH, ADASYN_TOMEK_DATA_FILE_PATH, \
    TESTING_DATA_FILE_PATH, RANDOM_STATE, N_JOBS, \
    SGD_ALPHA, SGD_MAX_ITER, SGD_LOSS, SGD_PENALTY, SGD_LEARNING_RATE, \
    LR_C, LR_PENALTY, LR_CLASS_WEIGHT, LR_VERBOSE, LR_MAX_ITER, LR_SOLVER, \
    RF_N_ESTIMATORS, RF_MAX_DEPTH, RF_MIN_SAMPLES_SPLIT, RF_MIN_SAMPLES_LEAF, RF_CLASS_WEIGHT, \
    NB_ALPHA, NB_TYPE, NB_BINARIZE, NB_VAR_SMOOTHING, \
    XGB_N_ESTIMATORS, XGB_MAX_DEPTH, XGB_LEARNING_RATE, XGB_SUBSAMPLE, XGB_COLSAMPLE_BY_TREE, \
    LGBM_N_ESTIMATORS, LGBM_MAX_DEPTH, LGBM_LEARNING_RATE, LGBM_NUM_LEAVES, LGBM_SUBSAMPLE, LGBM_COLSAMPLE_BY_TREE, \
    BEST_ACCURACY_MODEL_FILE_PATH, BEST_ACCURACY_MODEL_SCALER_FILE_PATH, \
    BEST_F1_MODEL_FILE_PATH, BEST_F1_MODEL_SCALER_FILE_PATH, \
    BEST_PRECISION_MODEL_FILE_PATH, BEST_PRECISION_MODEL_SCALER_FILE_PATH
from src.utils import load_data, save_data
from src.visualization import plot_roc_auc, plot_compare_models
from src.evaluate import compare_models
from src.models import (
    DiabetesSGDClassifier, DiabetesLogisticRegression, 
    DiabetesRandomForest, DiabetesNaiveBayes, 
    DiabetesXGBoostClassifier, DiabetesLightGBMClassifier
)

## **1. Data preparation**

### **1.1 Load balanced datasets**

In [12]:
# Load all 6 balanced datasets
ros_data = load_data(path=RANDOM_OVER_SAMPLING_DATA_FILE_PATH)
smote_data = load_data(path=SMOTE_DATA_FILE_PATH)
adasyn_data = load_data(path=ADASYN_DATA_FILE_PATH)
rus_data = load_data(path=RANDOM_UNDER_SAMPLING_DATA_FILE_PATH)
tomek_data = load_data(path=TOMEK_LINKS_DATA_FILE_PATH)
enn_data = load_data(path=EDITED_NEIGHBORS_DATA_FILE_PATH)
smote_tomek_data = load_data(path=SMOTE_TOMEK_LINKS_DATA_FILE_PATH)
smote_enn_data = load_data(path=SMOTE_ENN_DATA_FILE_PATH)
adasyn_tomek_data = load_data(path=ADASYN_TOMEK_DATA_FILE_PATH)

### **1.2 Split features and target**

In [13]:
# Split features and target of each dataset
# Random Oversampling
ros_X_train = ros_data["X"]
ros_y_train = ros_data["y"]
# SMOTE
smote_X_train = smote_data["X"]
smote_y_train = smote_data["y"]
# ADASYN
adasyn_X_train = adasyn_data["X"]
adasyn_y_train = adasyn_data["y"]
# Random Undersampling
rus_X_train = rus_data["X"]
rus_y_train = rus_data["y"]
# Tomek Links
tomek_X_train = tomek_data["X"]
tomek_y_train = tomek_data["y"]
# Edited Nearest Neighbors
enn_X_train = enn_data["X"]
enn_y_train = enn_data["y"]
# SMOTE + Tomek Links
smote_tomek_X_train = smote_tomek_data["X"]
smote_tomek_y_train = smote_tomek_data["y"]
# SMOTE + ENN
smote_enn_X_train = smote_enn_data["X"]
smote_enn_y_train = smote_enn_data["y"]
# ADASYN + Tomek Links
adasyn_tomek_X_train = adasyn_tomek_data["X"]
adasyn_tomek_y_train = adasyn_tomek_data["y"]

### **1.3 Normalize features for each dataset**

In [14]:
# Initialize MinMaxScaler for each dataset
ros_scaler = MinMaxScaler()
smote_scaler = MinMaxScaler()
adasyn_scaler = MinMaxScaler()
rus_scaler = MinMaxScaler()
tomek_scaler = MinMaxScaler()
enn_scaler = MinMaxScaler()
smote_tomek_scaler = MinMaxScaler()
smote_enn_scaler = MinMaxScaler()
adasyn_tomek_scaler = MinMaxScaler()

In [15]:
# Transform training dataset
ros_X_train_scaled = ros_scaler.fit_transform(ros_X_train)
smote_X_train_scaled = smote_scaler.fit_transform(smote_X_train)
adasyn_X_train_scaled = adasyn_scaler.fit_transform(adasyn_X_train)
rus_X_train_scaled = rus_scaler.fit_transform(rus_X_train)
tomek_X_train_scaled = tomek_scaler.fit_transform(tomek_X_train)
enn_X_train_scaled = enn_scaler.fit_transform(enn_X_train)
smote_tomek_X_train_scaled = smote_tomek_scaler.fit_transform(smote_tomek_X_train)
smote_enn_X_train_scaled = smote_enn_scaler.fit_transform(smote_enn_X_train)
adasyn_tomek_X_train_scaled = adasyn_tomek_scaler.fit_transform(adasyn_X_train)

### **1.4 Prepare test data**

In [16]:
# Load data
test_data = load_data(path=TESTING_DATA_FILE_PATH)

# Get testing features and target
X_test = test_data["X"]
y_test = test_data["y"]

In [17]:
# Apply normalization on test data
ros_X_test_scaled = ros_scaler.transform(X_test)
smote_X_test_scaled = smote_scaler.transform(X_test)
adasyn_X_test_scaled = adasyn_scaler.transform(X_test)
rus_X_test_scaled = rus_scaler.transform(X_test)
tomek_X_test_scaled = tomek_scaler.transform(X_test)
enn_X_test_scaled = enn_scaler.transform(X_test)
smote_tomek_X_test_scaled = smote_tomek_scaler.transform(X_test)
smote_enn_X_test_scaled = smote_enn_scaler.transform(X_test)
adasyn_tomek_X_test_scaled = adasyn_tomek_scaler.transform(X_test)

## **2. Model training by balancing method**

In [18]:
def train_evaluate_all_models(X_train, y_train, X_test, y_test, method_name):
    """
    Train and evaluate all 4 models on the given balanced dataset

    Parameters:
        X_train (np.ndarray): Training features
        y_train (np.ndarray): Training target
        X_test (np.ndarray): Testing features
        y_test (np.ndarray): Testing target
        method_name (str): Name of the balancing method

    Returns:
        results (dict): The evaluation results
        models (dict): The model's names and models
    """
    log_filename = method_name.lower().replace(" ", "_")
    log_filename = log_filename + "_method" if not "_method" in log_filename else log_filename
    log_file_path = f"{LOG_DIR}/5_{log_filename}.log"
    models = {}
    cv_scores = {}
    probabilities = {}

    # Initialize all models
    sgd_model = DiabetesSGDClassifier(
        loss=SGD_LOSS,
        penalty=SGD_PENALTY,
        alpha=SGD_ALPHA,
        max_iter=SGD_MAX_ITER,
        random_state=RANDOM_STATE,
        learning_rate=SGD_LEARNING_RATE,
        n_jobs=N_JOBS,
        log_file=log_file_path,
    )
    lr_model = DiabetesLogisticRegression(
        C=LR_C,
        penalty=LR_PENALTY,
        class_weight=LR_CLASS_WEIGHT,
        max_iter=LR_MAX_ITER,
        solver=LR_SOLVER,
        random_state=RANDOM_STATE,
        verbose=LR_VERBOSE,
        n_jobs=N_JOBS,
        log_file=log_file_path
    )
    nb_model = DiabetesNaiveBayes(
        nb_type=NB_TYPE,
        var_smoothing=NB_VAR_SMOOTHING,
        binarize=NB_BINARIZE,
        alpha=NB_ALPHA,
        log_file=log_file_path,
    )
    rf_model = DiabetesRandomForest(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_split=RF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        class_weight=RF_CLASS_WEIGHT,
        random_state=RANDOM_STATE,
        log_file=log_file_path,
    )
    xgb_model = DiabetesXGBoostClassifier(
        n_estimators=XGB_N_ESTIMATORS, 
        max_depth=XGB_MAX_DEPTH, 
        learning_rate=XGB_LEARNING_RATE, 
        subsample=XGB_SUBSAMPLE, 
        colsample_bytree=XGB_COLSAMPLE_BY_TREE,
        log_file=log_file_path,
    )
    lgbm_model = DiabetesLightGBMClassifier(
        n_estimators=LGBM_N_ESTIMATORS, 
        max_depth=LGBM_MAX_DEPTH, 
        learning_rate=LGBM_LEARNING_RATE, 
        num_leaves=LGBM_NUM_LEAVES, 
        subsample=LGBM_SUBSAMPLE, 
        colsample_bytree=LGBM_COLSAMPLE_BY_TREE,
        log_file=log_file_path, 
    )

    model_dict = {
        "SGDClassifier": sgd_model,
        "LogisticRegression": lr_model,
        "NaiveBayes": nb_model,
        "RandomForest": rf_model,
        "XGBoost": xgb_model, 
        "LightGBM": lgbm_model, 
    }

    # Train and evaluate each model
    for name, model in model_dict.items():
        print(f"Training {name} on {method_name} data...")

        # Train
        model.train(X=X_train, y=y_train)

        # Predict and evaluate
        y_pred = model.predict(X_test)
        model.evaluate(y_test, y_pred)

        # Get class probabilites
        probabilities[name] = model.predict_proba(X_test)

        # Cross-validation
        cv_result = model.cross_validate(X_train, y_train)
        cv_scores[name] = {
            'cv_mean': cv_result.mean(),
            'cv_std': cv_result.std()
        }

    # Get evaluation comparation results
    evaluation_results = compare_models(
        models_dict=model_dict,
        X_test=X_test,
        y_true=y_test
    )

    # Convert cv_scores dict to DataFrame
    cv_df = pd.DataFrame.from_dict(cv_scores, orient='index').reset_index()
    cv_df.columns = ['Model', 'CV Mean', 'CV Std']

    # Merge into evaluation results
    evaluation_results = pd.merge(evaluation_results, cv_df, on='Model', how='left')

    return evaluation_results, probabilities, models

### **2.1 Random Oversampling**

In [ ]:
# Train and get evaluation results with the Random OVersampling method
ros_evaluation_results, ros_class_probabilities, ros_models = train_evaluate_all_models(
    X_train=ros_X_train_scaled, 
    y_train=ros_y_train, 
    X_test=ros_X_test_scaled, 
    y_test=y_test, 
    method_name="Random Oversampling"
)

2025-09-17 16:42:10,728 - [src.models] - INFO - DiabetesSGDClassifier initialized with parameters:
2025-09-17 16:42:10,731 - [src.models] - INFO -   loss: log_loss
2025-09-17 16:42:10,732 - [src.models] - INFO -   penalty: l2
2025-09-17 16:42:10,734 - [src.models] - INFO -   alpha: 0.0001
2025-09-17 16:42:10,736 - [src.models] - INFO -   learning_rate: constant
2025-09-17 16:42:10,738 - [src.models] - INFO -   eta0: 0.01
2025-09-17 16:42:10,741 - [src.models] - INFO -   max_iter: 1000
2025-09-17 16:42:10,745 - [src.models] - INFO -   random_state: 42
2025-09-17 16:42:10,747 - [src.models] - INFO - DiabetesLogisticRegression initialized with parameters:
2025-09-17 16:42:10,751 - [src.models] - INFO -   C: 1.0
2025-09-17 16:42:10,753 - [src.models] - INFO -   penalty: l2
2025-09-17 16:42:10,759 - [src.models] - INFO -   class_weight: None
2025-09-17 16:42:10,762 - [src.models] - INFO -   solver: saga
2025-09-17 16:42:10,763 - [src.models] - INFO -   max_iter: 100
2025-09-17 16:42:10,765 

Training SGDClassifier on Random Oversampling data...


2025-09-17 16:42:14,175 - [src.models] - INFO - Model training completed. Training samples: 659553
2025-09-17 16:42:14,177 - [src.models] - INFO - Number of iterations: 7
2025-09-17 16:42:14,203 - [src.models] - INFO - Predictions made for 140504 samples
2025-09-17 16:42:14,405 - [src.models] - INFO - Model Evaluation Results:
2025-09-17 16:42:14,408 - [src.models] - INFO - Accuracy: 0.8294
2025-09-17 16:42:14,410 - [src.models] - INFO - Precision: 0.4652
2025-09-17 16:42:14,411 - [src.models] - INFO - Recall: 0.4463
2025-09-17 16:42:14,412 - [src.models] - INFO - F1-Score: 0.4527
2025-09-17 16:42:14,523 - [src.models] - INFO - Classification Report:
              precision    recall  f1-score   support

         0.0       0.87      0.94      0.90    114889
         1.0       0.00      0.00      0.00      3378
         2.0       0.52      0.40      0.46     22237

    accuracy                           0.83    140504
   macro avg       0.47      0.45      0.45    140504
weighted avg   

Training LogisticRegression on Random Oversampling data...


2025-09-17 16:42:41,944 - [src.models] - INFO - Model training completed. Training samples: 659553
2025-09-17 16:42:41,979 - [src.models] - INFO - Predictions made for 140504 samples
2025-09-17 16:42:42,123 - [src.models] - INFO - Model Evaluation Results:
2025-09-17 16:42:42,125 - [src.models] - INFO - Accuracy: 0.8344
2025-09-17 16:42:42,126 - [src.models] - INFO - Precision: 0.4736
2025-09-17 16:42:42,131 - [src.models] - INFO - Recall: 0.4421
2025-09-17 16:42:42,133 - [src.models] - INFO - F1-Score: 0.4519
2025-09-17 16:42:42,204 - [src.models] - INFO - Classification Report:
              precision    recall  f1-score   support

         0.0       0.87      0.95      0.91    114889
         1.0       0.00      0.00      0.00      3378
         2.0       0.55      0.38      0.45     22237

    accuracy                           0.83    140504
   macro avg       0.47      0.44      0.45    140504
weighted avg       0.80      0.83      0.81    140504

2025-09-17 16:42:42,276 - [src.m

Training NaiveBayes on Random Oversampling data...


2025-09-17 16:43:38,623 - [src.models] - INFO - Model training completed. Training samples: 659553
2025-09-17 16:43:38,624 - [src.models] - INFO - Number of classes: 3
2025-09-17 16:43:38,626 - [src.models] - INFO - Classes: [0. 1. 2.]
2025-09-17 16:43:38,687 - [src.models] - INFO - Predictions made for 140504 samples
2025-09-17 16:43:38,770 - [src.models] - INFO - Model Evaluation Results:
2025-09-17 16:43:38,771 - [src.models] - INFO - Accuracy: 0.7708
2025-09-17 16:43:38,773 - [src.models] - INFO - Precision: 0.4429
2025-09-17 16:43:38,775 - [src.models] - INFO - Recall: 0.4653
2025-09-17 16:43:38,776 - [src.models] - INFO - F1-Score: 0.4472
2025-09-17 16:43:38,880 - [src.models] - INFO - Classification Report:
              precision    recall  f1-score   support

         0.0       0.89      0.84      0.86    114889
         1.0       0.05      0.02      0.03      3378
         2.0       0.39      0.54      0.45     22237

    accuracy                           0.77    140504
   m

Training RandomForest on Random Oversampling data...


2025-09-17 16:45:07,590 - [src.models] - INFO - Model training completed. Training samples: 659553
2025-09-17 16:45:07,591 - [src.models] - INFO - Number of trees: 100
2025-09-17 16:45:10,959 - [src.models] - INFO - Predictions made for 140504 samples
2025-09-17 16:45:11,018 - [src.models] - INFO - Model Evaluation Results:
2025-09-17 16:45:11,019 - [src.models] - INFO - Accuracy: 0.7667
2025-09-17 16:45:11,020 - [src.models] - INFO - Precision: 0.5042
2025-09-17 16:45:11,021 - [src.models] - INFO - Recall: 0.6019
2025-09-17 16:45:11,022 - [src.models] - INFO - F1-Score: 0.5240
2025-09-17 16:45:11,086 - [src.models] - INFO - Classification Report:
              precision    recall  f1-score   support

         0.0       0.94      0.79      0.86    114889
         1.0       0.10      0.32      0.15      3378
         2.0       0.47      0.69      0.56     22237

    accuracy                           0.77    140504
   macro avg       0.50      0.60      0.52    140504
weighted avg      

In [ ]:
# The evaluation result of all models with the Random Oversampling method
ros_evaluation_results

In [ ]:
# Plot ROC and AUC of SGD model with Random Oversampling method
plot_roc_auc(
    y_true=y_test, 
    y_pred_proba=ros_class_probabilities["SGDClassifier"],
    model_name="SGD with Random Oversampling"
)

In [ ]:
# Plot ROC and AUC of Logistic Regression model with Random Oversampling method
plot_roc_auc(
    y_true=y_test, 
    y_pred_proba=ros_class_probabilities["LogisticRegression"],
    model_name="Logistic Regression with Random Oversampling"
)

In [ ]:
# Plot ROC and AUC of Naive Bayes model with Random Oversampling method
plot_roc_auc(
    y_true=y_test, 
    y_pred_proba=ros_class_probabilities["NaiveBayes"],
    model_name="Naive Bayes with Random Oversampling"
)

In [ ]:
# Plot ROC and AUC of Random Forest model with Random Oversampling method
plot_roc_auc(
    y_true=y_test, 
    y_pred_proba=ros_class_probabilities["RandomForest"],
    model_name="Random Forest with Random Oversampling"
)

### **2.2 SMOTE**

In [ ]:
# Train and get evaluation results with the SMOTE method
smote_evaluation_results, smote_class_probabilities, smote_models = train_evaluate_all_models(
    X_train=smote_X_train_scaled, 
    y_train=smote_y_train, 
    X_test=smote_X_test_scaled, 
    y_test=y_test, 
    method_name="SMOTE"
)

In [ ]:
# The evaluation result of all models with the SMOTE method
smote_evaluation_results

In [ ]:
# Plot ROC and AUC of SGD model with SMOTE method
plot_roc_auc(
    y_true=smote_y_train, 
    y_pred_proba=smote_class_probabilities["SGDClassifier"],
    model_name="SGD with SMOTE"
)

In [ ]:
# Plot ROC and AUC of Logistic Regression model with SMOTE method
plot_roc_auc(
    y_true=smote_y_train, 
    y_pred_proba=smote_class_probabilities["LogisticRegression"],
    model_name="Logistic Regression with SMOTE"
)

In [ ]:
# Plot ROC and AUC of Naive Bayes model with SMOTE method
plot_roc_auc(
    y_true=smote_y_train, 
    y_pred_proba=smote_class_probabilities["NaiveBayes"],
    model_name="Naive Bayes with SMOTE"
)

In [ ]:
# Plot ROC and AUC of Random Forest model with SMOTE method
plot_roc_auc(
    y_true=smote_y_train, 
    y_pred_proba=smote_class_probabilities["RandomForest"],
    model_name="Random Forest with SMOTE"
)

### **2.3 Random Undersampling**

In [ ]:
# Train and get evaluation results with the Random Undersamling method
rus_evaluation_results, rus_class_probabilities, rus_models = train_evaluate_all_models(
    X_train=rus_X_train_scaled, 
    y_train=rus_y_train, 
    X_test=rus_X_test_scaled, 
    y_test=y_test, 
    method_name="Random Undersampling"
)

In [ ]:
# The evaluation result of all models with the Random Undersampling method
rus_evaluation_results

In [ ]:
# Plot ROC and AUC of SGD model with Random Undersampling method
plot_roc_auc(
    y_true=rus_y_train, 
    y_pred_proba=rus_class_probabilities["SGDClassifier"],
    model_name="SGD with Random Undersampling"
)

In [ ]:
# Plot ROC and AUC of Logistic Regression model with Random Undersampling method
plot_roc_auc(
    y_true=rus_y_train, 
    y_pred_proba=rus_class_probabilities["LogisticRegression"],
    model_name="Logistic Regression with Random Undersampling"
)

In [ ]:
# Plot ROC and AUC of Naive Bayes model with Random Undersampling method
plot_roc_auc(
    y_true=rus_y_train, 
    y_pred_proba=rus_class_probabilities["NaiveBayes"],
    model_name="Naive Bayes with Random Undersampling"
)

In [ ]:
# Plot ROC and AUC of Random Forest model with Random Undersampling method
plot_roc_auc(
    y_true=rus_y_train, 
    y_pred_proba=rus_class_probabilities["RandomForest"],
    model_name="Random Forest with Random Undersampling"
)

### **2.4 Tomek Links**

In [ ]:
# Train and get evaluation results with the Tomek Links method
tomek_evaluation_results, tomek_class_probabilities, tomek_models = train_evaluate_all_models(
    X_train=tomek_X_train_scaled, 
    y_train=tomek_y_train, 
    X_test=tomek_X_test_scaled, 
    y_test=y_test, 
    method_name="Tomek Links"
)

In [ ]:
# The evaluation result of all models with the Tomek Links method
tomek_evaluation_results

In [ ]:
# Plot ROC and AUC of SGD model with Tomek Links method
plot_roc_auc(
    y_true=tomek_y_train, 
    y_pred_proba=tomek_class_probabilities["SGDClassifier"],
    model_name="SGD with Tomek Links"
)

In [ ]:
# Plot ROC and AUC of Logistic Regression model with Tomek Links method
plot_roc_auc(
    y_true=tomek_y_train, 
    y_pred_proba=tomek_class_probabilities["LogisticRegression"],
    model_name="Logistic Regression with Tomek Links"
)

In [ ]:
# Plot ROC and AUC of Naive Bayes model with Tomek Links method
plot_roc_auc(
    y_true=tomek_y_train, 
    y_pred_proba=tomek_class_probabilities["NaiveBayes"],
    model_name="Naive Bayes with Tomek Links"
)

In [ ]:
# Plot ROC and AUC of Random Forest model with Tomek Links method
plot_roc_auc(
    y_true=tomek_y_train, 
    y_pred_proba=tomek_class_probabilities["RandomForest"],
    model_name="Random Forest with Tomek Links"
)

### **2.5 SMOTE + Tomek Links**

In [ ]:
# Train and get evaluation results with the SMOTE + Tomek Links method
smote_tomek_evaluation_results, smote_tomek_class_probabilities, smote_tomek_models = train_evaluate_all_models(
    X_train=smote_tomek_X_train_scaled,
    y_train=smote_tomek_y_train, 
    X_test=smote_tomek_X_test_scaled, 
    y_test=y_test, 
    method_name="SMOTE + Tomek Links"
)

In [ ]:
# The evaluation result of all models with the SMOTE + Tomek Links method
smote_tomek_evaluation_results

In [ ]:
# Plot ROC and AUC of SGD model with SMOTE + Tomek Links method
plot_roc_auc(
    y_true=smote_tomek_y_train, 
    y_pred_proba=smote_tomek_class_probabilities["SGDClassifier"],
    model_name="SGD with SMOTE + Tomek Links"
)

In [ ]:
# Plot ROC and AUC of Logistic Regression model with SMOTE + Tomek Links method
plot_roc_auc(
    y_true=smote_tomek_y_train, 
    y_pred_proba=smote_tomek_class_probabilities["LogisticRegression"],
    model_name="Logistic Regression with SMOTE + Tomek Links"
)

In [ ]:
# Plot ROC and AUC of Naive Bayes model with SMOTE + Tomek Links method
plot_roc_auc(
    y_true=smote_tomek_y_train, 
    y_pred_proba=smote_tomek_class_probabilities["NaiveBayes"],
    model_name="Naive Bayes with SMOTE + Tomek Links"
)

In [ ]:
# Plot ROC and AUC of Random Forest model with SMOTE + Tomek Links method
plot_roc_auc(
    y_true=smote_tomek_y_train, 
    y_pred_proba=smote_tomek_class_probabilities["RandomForest"],
    model_name="Random Forest with SMOTE + Tomek Links"
)

### **2.6 SMOTE + ENN**

In [ ]:
# Train and get evaluation results with the SMOTE + ENN method
smote_enn_evaluation_results, smote_enn_class_probabilities, smote_enn_models = train_evaluate_all_models(
    X_train=smote_enn_X_train_scaled,
    y_train=smote_enn_y_train, 
    X_test=smote_enn_X_test_scaled, 
    y_test=y_test, 
    method_name="SMOTE + ENN"
)

In [ ]:
# The evaluation result of all models with the SMOTE + ENN method
smote_enn_evaluation_results

In [ ]:
# Plot ROC and AUC of SGD Classifier model with SMOTE + ENN method
plot_roc_auc(
    y_true=smote_enn_y_train, 
    y_pred_proba=smote_enn_class_probabilities["SGDClassifier"],
    model_name="SGD with SMOTE + ENN"
)

In [ ]:
# Plot ROC and AUC of Logistic Regression model with SMOTE + ENN method
plot_roc_auc(
    y_true=smote_enn_y_train, 
    y_pred_proba=smote_enn_class_probabilities["LogisticRegression"],
    model_name="Logistic Regression with SMOTE + ENN"
)

In [ ]:
# Plot ROC and AUC of Naive Bayes model with SMOTE + ENN method
plot_roc_auc(
    y_true=smote_enn_y_train, 
    y_pred_proba=smote_enn_class_probabilities["NaiveBayes"],
    model_name="Naive Bayes with SMOTE + ENN"
)

In [ ]:
# Plot ROC and AUC of Random Forest model with SMOTE + ENN method
plot_roc_auc(
    y_true=smote_enn_y_train, 
    y_pred_proba=smote_enn_class_probabilities["RandomForest"],
    model_name="Random Forest with SMOTE + ENN"
)

## **3. Performance visualization**

In [ ]:
# Plot comprehensive comparison of models with Random Oversampling
plot_compare_models(comparison_df=ros_evaluation_results, title="Model Performance Comparison on Random Oversampling method")

In [ ]:
# Plot comprehensive comparison of models with SMOTE
plot_compare_models(comparison_df=smote_evaluation_results, title="Model Performance Comparison on SMOTE method")

In [ ]:
# Plot comprehensive comparison of models with Random Under Sampling
plot_compare_models(comparison_df=rus_evaluation_results, title="Model Performance Comparison on Random Undersampling method")

In [ ]:
# Plot comprehensive comparison of models with Tomek Links
plot_compare_models(comparison_df=tomek_evaluation_results, title="Model Performance Comparison on Tomek Links method")

In [ ]:
# Plot comprehensive comparison of models with SMOTE + Tomek Links
plot_compare_models(comparison_df=smote_tomek_evaluation_results, title="Model Performance Comparison on SMOTE + Tomek Links method")

In [ ]:
# Plot comprehensive comparison of models with SMOTE + ENN
plot_compare_models(comparison_df=smote_enn_evaluation_results, title="Model Performance Comparison on SMOTE + ENN method")

## **4. Comprehensive evaluation**

In [ ]:
ros_evaluation_results["Method"] = "Random Oversampling"
smote_evaluation_results["Method"] = "SMOTE"
rus_evaluation_results["Method"] = "Random Undersampling"
tomek_evaluation_results["Method"] = "Tomek Links"
smote_tomek_evaluation_results["Method"] = "SMOTE + Tomek Links"
smote_enn_evaluation_results["Method"] = "SMOTE + ENN"

comparison_df = pd.concat([
    ros_evaluation_results,
    smote_evaluation_results,
    rus_evaluation_results,
    tomek_evaluation_results,
    smote_tomek_evaluation_results,
    smote_enn_evaluation_results
], ignore_index=True)

In [ ]:
best_accuracy = comparison_df.loc[comparison_df['Accuracy'].idxmax()]
best_f1      = comparison_df.loc[comparison_df['F1-Score'].idxmax()]
best_recall  = comparison_df.loc[comparison_df['Recall'].idxmax()]
best_precision = comparison_df.loc[comparison_df['Precision'].idxmax()]

In [ ]:
print(f"Best Accuracy: {best_accuracy['Model']} with {best_accuracy['Method']} - {best_accuracy['Accuracy']:.4f}")
print(f"Best F1-Score: {best_f1['Model']} with {best_f1['Method']} - {best_f1['F1-Score']:.4f}")
print(f"Best Recall: {best_recall['Model']} with {best_recall['Method']} - {best_recall['Recall']:.4f}")
print(f"Best Precision: {best_precision['Model']} with {best_precision['Method']} - {best_precision['Precision']:.4f}")

In [ ]:
best_overall = comparison_df.loc[comparison_df['F1-Score'].idxmax()]
print(f"RECOMMENDED MODEL: {best_overall['Model']} with {best_overall['Method']}")
print(f" - Accuracy: {best_overall['Accuracy']:.4f}")
print(f" - Precision: {best_overall['Precision']:.4f}")
print(f" - Recall: {best_overall['Recall']:.4f}")
print(f" - F1-Score: {best_overall['F1-Score']:.4f}")

## **5. Save the best models**

In [ ]:
# Save the scaler and model have the best accuracy
save_data(
    path=BEST_ACCURACY_MODEL_SCALER_FILE_PATH,
    data=tomek_scaler
)
tomek_models["LogisticRegression"].save_model(model_path=BEST_ACCURACY_MODEL_FILE_PATH)

In [ ]:
# Save the scaler and model have the best F1-score
save_data(
    path=BEST_F1_MODEL_SCALER_FILE_PATH,
    data=tomek_scaler
)
smote_enn_models["RandomForest"].save_model(model_path=BEST_F1_MODEL_FILE_PATH)

In [ ]:
# Save the scaler and model have the best recall
save_data(
    path=BEST_F1_MODEL_SCALER_FILE_PATH,
    data=tomek_scaler
)
rus_models["LogisticRegression"].save_model(model_path=f"{MODEL_DIR}/logistic_regression_model_best_recall.pkl")